# Feature Extraction Pipeline 

In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import cv2
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from skimage.feature import graycomatrix, graycoprops

import sys
sys.path.append("..")
from utils.metrics import evaluate_model, compare_splits

SEED = 67
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
DATA_DIR   = "../dataset/processed"
MODEL_PATH = "../models"
FEAT_PATH  = "../features/ensemble"

TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR   = os.path.join(DATA_DIR, "val")
TEST_DIR  = os.path.join(DATA_DIR, "test")

os.makedirs(FEAT_PATH, exist_ok=True)

IMG_SIZE   = 224
BATCH_SIZE = 32

# Loading Data 

In [3]:
transform = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)),transforms.ToTensor()])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=transform)
val_dataset   = datasets.ImageFolder(VAL_DIR,   transform=transform)
test_dataset  = datasets.ImageFolder(TEST_DIR,  transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

NUM_CLASSES = len(train_dataset.classes)
print("Classes :", NUM_CLASSES)
print("Class names:", train_dataset.classes)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")

Classes : 38
Class names: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_

# Loading Models

In [4]:
mobilenet = models.mobilenet_v3_small()

in_features = mobilenet.classifier[3].in_features
mobilenet.classifier[3] = nn.Linear(in_features, NUM_CLASSES)

mobilenet.load_state_dict(torch.load(os.path.join(MODEL_PATH, "mobilenetv3.pth"), map_location=device))

mobilenet = mobilenet.to(device)
mobilenet.eval()
print("MobileNetV3 loaded")

MobileNetV3 loaded


In [5]:
googlenet = models.googlenet(aux_logits=False)

in_features = googlenet.fc.in_features
googlenet.fc = nn.Linear(in_features, NUM_CLASSES)

googlenet.load_state_dict(torch.load(os.path.join(MODEL_PATH, "googlenet.pth"), map_location=device))

googlenet = googlenet.to(device)
googlenet.eval()
print("GoogLeNet loaded")

GoogLeNet loaded


/home/rishi_677/Plant_Disease_DAC_Project/venv/lib/python3.12/site-packages/torchvision/models/googlenet.py:47: FutureWarning: The default weight initialization of GoogleNet will be changed in future releases of torchvision. If you wish to keep the old behavior (which leads to long initialization times due to scipy/scipy#11299), please set init_weights=True.
  warnings.warn(


In [6]:
convnext = models.convnext_small()

in_features = convnext.classifier[2].in_features
convnext.classifier[2] = nn.Linear(in_features, NUM_CLASSES)

convnext.load_state_dict(torch.load(os.path.join(MODEL_PATH, "convnext_small.pth"), map_location=device))

convnext = convnext.to(device)
convnext.eval()
print("ConvNeXtSmall loaded")

ConvNeXtSmall loaded


In [7]:
def get_glcm_features(img):
    
    img = img.permute(1, 2, 0).cpu().numpy()
    img = (img * 255).astype(np.uint8)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    glcm = graycomatrix(
        gray,
        distances=[1, 2],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=256,
        symmetric=True,
        normed=True
    )

    contrast    = graycoprops(glcm, "contrast").flatten()     
    homogeneity = graycoprops(glcm, "homogeneity").flatten()  
    energy      = graycoprops(glcm, "energy").flatten()       
    correlation = graycoprops(glcm, "correlation").flatten()  

    return np.hstack([contrast, homogeneity, energy, correlation])  

In [8]:
def extract_all_features(loader):
    glcm_feat   = []
    mobile_feat = []
    google_feat = []
    conv_feat   = []
    labels      = []

    with torch.no_grad():
        for images, lbl in loader:
            images = images.to(device)

            for img in images:
                glcm_feat.append(get_glcm_features(img))

            x1 = mobilenet.features(images)
            x1 = mobilenet.avgpool(x1)
            x1 = torch.flatten(x1, 1)   

            x2 = googlenet.conv1(images)
            x2 = googlenet.maxpool1(x2)
            x2 = googlenet.conv2(x2)
            x2 = googlenet.conv3(x2)
            x2 = googlenet.maxpool2(x2)
            x2 = googlenet.inception3a(x2)
            x2 = googlenet.inception3b(x2)
            x2 = googlenet.maxpool3(x2)
            x2 = googlenet.inception4a(x2)
            x2 = googlenet.inception4b(x2)
            x2 = googlenet.inception4c(x2)
            x2 = googlenet.inception4d(x2)
            x2 = googlenet.inception4e(x2)
            x2 = googlenet.maxpool4(x2)
            x2 = googlenet.inception5a(x2)
            x2 = googlenet.inception5b(x2)
            x2 = googlenet.avgpool(x2)
            x2 = torch.flatten(x2, 1)           

            x3 = convnext.features(images)
            x3 = convnext.avgpool(x3)
            x3 = torch.flatten(x3, 1)           

            mobile_feat.append(x1.cpu().numpy())
            google_feat.append(x2.cpu().numpy())
            conv_feat.append(x3.cpu().numpy())
            labels.append(lbl.numpy())

    glcm_feat   = np.array(glcm_feat)           
    mobile_feat = np.vstack(mobile_feat)         
    google_feat = np.vstack(google_feat)         
    conv_feat   = np.vstack(conv_feat)           
    labels      = np.hstack(labels)             
    
    return glcm_feat, mobile_feat, google_feat, conv_feat, labels

# Extracting the Features 

In [9]:
print("Extracting TRAIN features...")
train_glcm, train_mobile, train_google, train_conv, train_labels = extract_all_features(train_loader)

print(f"  GLCM        : {train_glcm.shape}")
print(f"  MobileNetV3 : {train_mobile.shape}")
print(f"  GoogLeNet   : {train_google.shape}")
print(f"  ConvNeXt    : {train_conv.shape}")
print(f"  Labels      : {train_labels.shape}")

Extracting TRAIN features...
  GLCM        : (65891, 32)
  MobileNetV3 : (65891, 576)
  GoogLeNet   : (65891, 1024)
  ConvNeXt    : (65891, 768)
  Labels      : (65891,)


In [10]:
print("Extracting VAL features...")
val_glcm, val_mobile, val_google, val_conv, val_labels = extract_all_features(val_loader)

print(f"  GLCM        : {val_glcm.shape}")
print(f"  MobileNetV3 : {val_mobile.shape}")
print(f"  GoogLeNet   : {val_google.shape}")
print(f"  ConvNeXt    : {val_conv.shape}")
print(f"  Labels      : {val_labels.shape}")

Extracting VAL features...
  GLCM        : (4380, 32)
  MobileNetV3 : (4380, 576)
  GoogLeNet   : (4380, 1024)
  ConvNeXt    : (4380, 768)
  Labels      : (4380,)


In [11]:
print("Extracting TEST features...")
test_glcm, test_mobile, test_google, test_conv, test_labels = extract_all_features(test_loader)

print(f"  GLCM        : {test_glcm.shape}")
print(f"  MobileNetV3 : {test_mobile.shape}")
print(f"  GoogLeNet   : {test_google.shape}")
print(f"  ConvNeXt    : {test_conv.shape}")
print(f"  Labels      : {test_labels.shape}")

Extracting TEST features...
  GLCM        : (17596, 32)
  MobileNetV3 : (17596, 576)
  GoogLeNet   : (17596, 1024)
  ConvNeXt    : (17596, 768)
  Labels      : (17596,)


In [12]:
X_train = np.hstack([train_glcm, train_mobile, train_google, train_conv])  
X_val   = np.hstack([val_glcm,   val_mobile,   val_google,   val_conv  ])
X_test  = np.hstack([test_glcm,  test_mobile,  test_google,  test_conv ])

y_train = train_labels
y_val   = val_labels
y_test  = test_labels

print("Fused feature shapes:")
print(f"  X_train : {X_train.shape}")
print(f"  X_val   : {X_val.shape}")
print(f"  X_test  : {X_test.shape}")

Fused feature shapes:
  X_train : (65891, 2400)
  X_val   : (4380, 2400)
  X_test  : (17596, 2400)


In [13]:
np.save(os.path.join(FEAT_PATH, "X_train.npy"), X_train)
np.save(os.path.join(FEAT_PATH, "X_val.npy"),   X_val)
np.save(os.path.join(FEAT_PATH, "X_test.npy"),  X_test)

np.save(os.path.join(FEAT_PATH, "y_train.npy"), y_train)
np.save(os.path.join(FEAT_PATH, "y_val.npy"),   y_val)
np.save(os.path.join(FEAT_PATH, "y_test.npy"),  y_test)

print("Saved to:", FEAT_PATH)
print("Files:", os.listdir(FEAT_PATH))

Saved to: ../features/ensemble
Files: ['X_test.npy', 'y_val.npy', 'X_val.npy', 'X_train.npy', 'y_test.npy', 'y_train.npy']
